<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/notebooks/stage_06_window_scaling_seq2one/stage_06_window_scaling_seq2one.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **stage_06_window_scaling_seq2one**




## **0. Configuración del Entorno**


### 0.1. Acceso a Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')
drive_path = "/content/drive/MyDrive/neural_profit"

Mounted at /content/drive


### 0.2. Instalación de librerías


In [2]:
#!{sys.executable} -m pip install -q ta
#print("Librería instalada: technical-analysis")

### 0.3. Importación de librerías


In [3]:
import sys
import re
#Instalación de librería pandas_market_calendars
#!{sys.executable} -m pip install -q pandas_market_calendars
#print("Librería instalada: pandas_market_calendars")


from functools import reduce
# Utilidades generales
from datetime import datetime, timedelta
import os
import glob
import requests
import warnings
warnings.filterwarnings('ignore')

# Manejo y procesamiento de datos
#import ta
import pandas as pd
import numpy as np
from tabulate import tabulate
import matplotlib.pyplot as plt
# Calendario de mercados
#import pandas_market_calendars as mcal

#from ta.momentum import StochasticOscillator, ROCIndicator
#from ta.volatility import BollingerBands, AverageTrueRange

from scipy.stats import spearmanr
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import joblib
import os
import json
import logging
from pathlib import Path
from typing import Dict, Any, List, Tuple

import numpy as np
import pandas as pd

# ----------------------------
# Logging
# ----------------------------
logging.basicConfig(
    level=os.environ.get("LOG_LEVEL", "INFO"),
    format="%(asctime)s | %(levelname)s | %(message)s",
)
log = logging.getLogger("stage_06_window_scaling_seq2seq")

### 0.4. Definición de rutas

In [4]:
# ============================================================
# Paths / IO (via env o defaults)
# ============================================================
DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

IN_PARQUET_TRAIN = Path(os.environ.get("IN_PARQUET_TRAIN", "data/splits/mnq_train.parquet"))
IN_PARQUET_VALID = Path(os.environ.get("IN_PARQUET_VALID", "data/splits/mnq_valid.parquet"))
IN_PARQUET_TEST = Path(os.environ.get("IN_PARQUET_TEST", "data/splits/mnq_test.parquet"))

IN_ARTIFACT = Path(os.environ.get("IN_ARTIFACT", "reports/stage_04_feature_engineering_summary.json"))
IN_ARTIFACT_GESTATION = Path(os.environ.get("IN_ARTIFACT_GESTATION", "reports/stage_03b_target_definition_summary.json"))



OUT_WINDOWS_60_TRAIN = Path(os.environ.get("OUT_WINDOWS_60", "data/windows/windows_train_60.npz"))
OUT_WINDOWS_60_VALID = Path(os.environ.get("OUT_WINDOWS_60", "data/windows/windows_valid_60.npz"))
OUT_WINDOWS_60_TEST  = Path(os.environ.get("OUT_WINDOWS_60", "data/windows/windows_test_60.npz"))

OUT_WINDOWS_90_TRAIN = Path(os.environ.get("OUT_WINDOWS_90", "data/windows/windows_train_90.npz"))
OUT_WINDOWS_90_VALID = Path(os.environ.get("OUT_WINDOWS_90", "data/windows/windows_valid_90.npz"))
OUT_WINDOWS_90_TEST  = Path(os.environ.get("OUT_WINDOWS_90", "data/windows/windows_test_90.npz"))

# Escalados

OUT_WINDOWS_60_TRAIN_Z = Path(os.environ.get("OUT_WINDOWS_60_Z", "data/windows/scaled/windows_train_60_z.npz"))
OUT_WINDOWS_60_VALID_Z = Path(os.environ.get("OUT_WINDOWS_60_Z", "data/windows/scaled/windows_valid_60_z.npz"))
OUT_WINDOWS_60_TEST_Z  = Path(os.environ.get("OUT_WINDOWS_60_Z", "data/windows/scaled/windows_test_60_z.npz"))

OUT_WINDOWS_90_TRAIN_Z = Path(os.environ.get("OUT_WINDOWS_90_Z", "data/windows/scaled/windows_train_90_z.npz"))
OUT_WINDOWS_90_VALID_Z = Path(os.environ.get("OUT_WINDOWS_90_Z", "data/windows/scaled/windows_valid_90_z.npz"))
OUT_WINDOWS_90_TEST_Z  = Path(os.environ.get("OUT_WINDOWS_90_Z", "data/windows/scaled/windows_test_90_z.npz"))

OUT_SCALER_60 = Path(os.environ.get("OUT_SCALER", "data/windows/scaled/scaler_60.pkl"))
OUT_SCALER_90 = Path(os.environ.get("OUT_SCALER", "data/windows/scaled/scaler_90.pkl"))

OUT_SUMMARY = Path(os.environ.get("OUT_SUMMARY", "reports/windows_scaled_summary.json"))


#PARA NOTEBOOK

IN_PARQUET_TRAIN = DRIVE_DIR / IN_PARQUET_TRAIN
IN_PARQUET_VALID = DRIVE_DIR / IN_PARQUET_VALID
IN_PARQUET_TEST = DRIVE_DIR / IN_PARQUET_TEST
IN_ARTIFACT = DRIVE_DIR / IN_ARTIFACT
IN_ARTIFACT_GESTATION = DRIVE_DIR / IN_ARTIFACT_GESTATION
OUT_WINDOWS_60_TRAIN = DRIVE_DIR / OUT_WINDOWS_60_TRAIN
OUT_WINDOWS_60_VALID = DRIVE_DIR / OUT_WINDOWS_60_VALID
OUT_WINDOWS_60_TEST = DRIVE_DIR / OUT_WINDOWS_60_TEST

OUT_WINDOWS_90_TRAIN = DRIVE_DIR / OUT_WINDOWS_90_TRAIN
OUT_WINDOWS_90_VALID = DRIVE_DIR / OUT_WINDOWS_90_VALID
OUT_WINDOWS_90_TEST = DRIVE_DIR / OUT_WINDOWS_90_TEST

OUT_WINDOWS_60_TRAIN_Z = DRIVE_DIR / OUT_WINDOWS_60_TRAIN_Z
OUT_WINDOWS_60_VALID_Z = DRIVE_DIR / OUT_WINDOWS_60_VALID_Z
OUT_WINDOWS_60_TEST_Z = DRIVE_DIR / OUT_WINDOWS_60_TEST_Z

OUT_WINDOWS_90_TRAIN_Z = DRIVE_DIR / OUT_WINDOWS_90_TRAIN_Z
OUT_WINDOWS_90_VALID_Z = DRIVE_DIR / OUT_WINDOWS_90_VALID_Z
OUT_WINDOWS_90_TEST_Z = DRIVE_DIR / OUT_WINDOWS_90_TEST_Z

OUT_SCALER_60 = DRIVE_DIR / OUT_SCALER_60
OUT_SCALER_90 = DRIVE_DIR / OUT_SCALER_90

OUT_SUMMARY = DRIVE_DIR / OUT_SUMMARY



# **1. Carga de datos**

## 1.1. Carga de datasets `mnq_train`, `mnq_valid` y `mnq_test`






In [5]:
def load_mnq_parquet(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el parquet de entrada: {path}")

    log.info("[OK] Cargando parquet: %s", path)

    df = pd.read_parquet(path)

    # Asegurar DatetimeIndex
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index)

    # Orden temporal explícito
    df = df.sort_index()

    return df


In [7]:
mnq_train = load_mnq_parquet(IN_PARQUET_TRAIN)
mnq_valid = load_mnq_parquet(IN_PARQUET_VALID)
mnq_test = load_mnq_parquet(IN_PARQUET_TEST)

## **1.2. Información de datasets**


In [8]:
def info_dataset(df, df_name):
    print(f"Información del dataset: {df_name}\n")

    # -------------------------
    # Días y registros
    # -------------------------
    num_dias = df["date"].nunique()
    print(f"\tCantidad de días: {num_dias}")

    validos_por_dia = (
        df.dropna(subset=["close"])
          .groupby("date")
          .size()
    )
    promedio_por_fecha = validos_por_dia.mean()
    print(f"\tRegistros por día: {int(promedio_por_fecha)}")

    # -------------------------
    # Horarios (desde minute_of_day)
    # -------------------------
    if "minute_of_day" in df.columns:
        min_minute = int(df["minute_of_day"].min())
        max_minute = int(df["minute_of_day"].max())

        primer_hora = f"{min_minute // 60:02d}:{min_minute % 60:02d}"
        ultima_hora = f"{max_minute // 60:02d}:{max_minute % 60:02d}"
    else:
        primer_hora = None
        ultima_hora = None

    print(f"\tHora diaria de inicio: {primer_hora}")
    print(f"\tHora diaria de final: {ultima_hora}")

    # -------------------------
    # Columnas del dataset
    # -------------------------
    print("\n\tColumnas del dataset:")
    for c in df.columns:
        print(f"\t- {c}")

    return num_dias

In [9]:
info_dataset(mnq_train, 'mnq_train')
info_dataset(mnq_valid, 'mnq_valid')
info_dataset(mnq_test, 'mnq_test')

Información del dataset: mnq_train

	Cantidad de días: 912
	Registros por día: 421
	Hora diaria de inicio: 07:30
	Hora diaria de final: 14:30

	Columnas del dataset:
	- date
	- minute_of_day
	- close
	- ema60
	- roc60
	- roc30
	- roc30_active
	- roc20
	- roc20_active
	- mom10
	- mom10_active
	- mom5
	- mom5_active
	- mom3
	- mom3_active
	- mom5_mom10
	- mom5_mom10_active
	- mom3_mom10
	- mom3_mom10_active
	- mom3_mom5
	- mom3_mom5_active
	- ret60
	- ret90
Información del dataset: mnq_valid

	Cantidad de días: 195
	Registros por día: 421
	Hora diaria de inicio: 07:30
	Hora diaria de final: 14:30

	Columnas del dataset:
	- date
	- minute_of_day
	- close
	- ema60
	- roc60
	- roc30
	- roc30_active
	- roc20
	- roc20_active
	- mom10
	- mom10_active
	- mom5
	- mom5_active
	- mom3
	- mom3_active
	- mom5_mom10
	- mom5_mom10_active
	- mom3_mom10
	- mom3_mom10_active
	- mom3_mom5
	- mom3_mom5_active
	- ret60
	- ret90
Información del dataset: mnq_test

	Cantidad de días: 196
	Registros por día: 42

196

# **2. Carga de features para cada horizonte 60 y 90min**

Definimos el target de cada horizonte:

In [10]:
start_minute_full_day = 450
end_minute_full_day = 870

start_minute_gestation = 480
final_minute_gestation = 540

start_minute_execution = 540
final_minute_execution = 600

In [13]:
targets = ['ret60', 'ret90']

In [14]:
target_60 = targets[0]
target_90 = targets[1]

In [15]:
features = [
 'minute_of_day',
 'close',
 'ema60',
 'roc60',
 'roc30',
 'roc20',
 'mom10',
 'mom5',
 'mom3',
 'mom5_mom10',
 'mom3_mom10',
 'mom3_mom5',
   ]

In [16]:
len(features)

12

In [17]:
features_flags = [
 'roc30_active',
 'roc20_active',
 'mom10_active',
 'mom5_active',
 'mom3_active',
 'mom5_mom10_active',
 'mom3_mom10_active',
 'mom3_mom5_active'
    ]

In [18]:
len(features)+len(features_flags)

20

# **4. Definición de windows size**

El `window_size` está condicionado por el feature que más historial necesita, en nuestro caso `roc_60`, necesitan 60 minutos previos para poder calcular su primer valor válido.

Si hacemos más corto el window_size corremos el riesgo de perder información o generar NaNs.


In [19]:
windows_size = 60

# **5. Escalado de los datasets**

## **5.1. Introducción teórica**

El escalado es una etapa **crítica** del pipeline, ya que los modelos de *machine learning* son sensibles a la **escala relativa de las variables de entrada**.  
En este proyecto conviven distintos tipos de variables, entre ellas:

- precios,
- indicadores técnicos,
- interacciones entre indicadores,
- variables temporales,
- y flags binarios,

cada una con **rangos y distribuciones muy diferentes**.

Por este motivo, el escalado debe realizarse de forma **controlada y consistente**, respetando tanto la naturaleza de cada feature como la **coherencia temporal** del dataset.

### **5.1.1. Principios que guían el escalado**


1. **Evitar data leakage**  
   El *scaler* se ajusta (*fit*) **exclusivamente con el conjunto de entrenamiento** y luego se aplica (*transform*) a los conjuntos de validación y test.  
   De este modo se evita introducir información futura durante el entrenamiento.

2. **Escalar solo variables continuas**  
   Se escalan:
   - precios,
   - indicadores técnicos,
   - interacciones,
   - la variable temporal `minute_of_day`.

   No se escalan:
   - flags `_active` (variables binarias),
   - columnas de fecha o identificadores.

3. **Escalar antes de generar las ventanas**  
   El escalado se aplica cuando los datos aún están en formato tabular (`DataFrame`), preservando los nombres de las columnas.  
   Esto permite:
   - seleccionar explícitamente qué columnas se escalan,
   - mantener trazabilidad sobre las features,
   - evitar errores silenciosos una vez que las ventanas se vectorizan y se pierde la referencia a los nombres.

### **5.1.2. Objetivo del escalado**



El objetivo del escalado no es alterar la información contenida en las features, sino **proyectarlas a un espacio numérico comparable**, facilitando que el modelo:

- aprenda relaciones estables entre variables,
- no priorice artificialmente features por su magnitud,
- y generalice correctamente entre distintos días y regímenes intradía.

Con estos criterios establecidos, el siguiente paso consiste en implementar el escalado de forma **explícita y reproducible** para los conjuntos `mnq_train`, `mnq_valid` y `mnq_test`.


## **5.2. Implementación de escalado**

### **5.2.1. Función para elegir escalador**


In [20]:
from __future__ import annotations
from typing import Literal, Optional
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

def choose_scaler(
    scaler_type: str = "standard",
    *,
    with_mean: bool = True,
    with_std: bool = True,
    feature_range: tuple[float, float] = (0.0, 1.0),
    quantile_range: tuple[float, float] = (25.0, 75.0),
) -> Optional[object]:
    """
    Devuelve un scaler de sklearn según `scaler_type`.

    scaler_type soportados:
      - 'standard' | 'z' | 'zscore' -> StandardScaler
      - 'minmax'   | 'min_max'      -> MinMaxScaler
      - 'robust'                   -> RobustScaler
      - 'none' | 'passthrough'     -> None (sin escalado)

    Retorna:
      - instancia de scaler (fit/transform) o None si no se desea escalar.
    """
    st = (scaler_type or "").strip().lower()

    if st in {"standard", "z", "zscore"}:
        return StandardScaler(with_mean=with_mean, with_std=with_std)

    if st in {"minmax", "min_max"}:
        return MinMaxScaler(feature_range=feature_range)

    if st in {"robust"}:
        return RobustScaler(quantile_range=quantile_range, with_centering=True, with_scaling=True)

    if st in {"none", "passthrough"}:
        return None

    raise ValueError(
        f"scaler_type inválido: '{scaler_type}'. "
        "Use: 'standard', 'minmax', 'robust' o 'none'."
    )


### **5.2.2. Función para escalar datasets train, valid y test**


In [21]:
import pandas as pd
from typing import Tuple, List, Dict, Any

def scale_mnq_splits(
    mnq_train: pd.DataFrame,
    mnq_valid: pd.DataFrame,
    mnq_test: pd.DataFrame,
    *,
    scaler,
    date_col: str = "date",
    target_cols: Tuple[str, str] = ("delta60", "delta90"),
    flag_suffix: str = "_active",
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, Dict[str, Any]]:
    """
    Escala los splits mnq_train/mnq_valid/mnq_test evitando data leakage.

    Reglas:
      - Fit del scaler SOLO en mnq_train.
      - Escala únicamente columnas numéricas continuas (incluye minute_of_day).
      - NO escala flags *_active.
      - NO escala targets (delta60/delta90).
      - NO escala ni altera la columna date.

    Retorna:
      - mnq_train_scaled, mnq_valid_scaled, mnq_test_scaled
      - metadata con columnas usadas y parámetros básicos
    """
    # Validación columnas iguales (recomendado)
    cols_train = list(mnq_train.columns)
    if list(mnq_valid.columns) != cols_train or list(mnq_test.columns) != cols_train:
        raise ValueError("mnq_train, mnq_valid y mnq_test deben tener exactamente las mismas columnas y orden.")

    # Identificar columnas
    flag_cols = [c for c in cols_train if c.endswith(flag_suffix)]
    target_cols = list(target_cols)

    # Columnas a excluir del escalado
    exclude = set([date_col]) | set(flag_cols) | set(target_cols)

    # Columnas numéricas a escalar (features continuas)
    scale_cols = [c for c in cols_train if c not in exclude]

    if scaler is None:
        # Sin escalado: devolver copias
        meta = {
            "scaled": False,
            "scale_cols": scale_cols,
            "flag_cols": flag_cols,
            "target_cols": target_cols,
            "date_col": date_col,
        }
        return mnq_train.copy(), mnq_valid.copy(), mnq_test.copy(), meta

    # Copias
    tr = mnq_train.copy()
    va = mnq_valid.copy()
    te = mnq_test.copy()

    # Fit SOLO en train
    scaler.fit(tr[scale_cols])

    # Transform en todos
    tr.loc[:, scale_cols] = scaler.transform(tr[scale_cols])
    va.loc[:, scale_cols] = scaler.transform(va[scale_cols])
    te.loc[:, scale_cols] = scaler.transform(te[scale_cols])

    meta = {
        "scaled": True,
        "scaler_class": scaler.__class__.__name__,
        "scale_cols": scale_cols,
        "flag_cols": flag_cols,
        "target_cols": target_cols,
        "date_col": date_col,
    }
    return tr, va, te, meta


### **5.2.3. Guardar dataset escalados**


In [22]:
import os
import json
import pandas as pd
from typing import Dict, Any

def save_scaled_datasets(
    *,
    mnq_train_scaled: pd.DataFrame,
    mnq_valid_scaled: pd.DataFrame,
    mnq_test_scaled: pd.DataFrame,
    scale_meta: Dict[str, Any],
    base_path: str = "/content/drive/MyDrive/neural_profit/data/scaled",
    file_format: str = "parquet",
) -> None:
    """
    Guarda los datasets escalados y la metadata de escalado en disco.

    Estructura de salida:
      base_path/
        ├── mnq_train_scaled.parquet
        ├── mnq_valid_scaled.parquet
        ├── mnq_test_scaled.parquet
        └── scaling_meta.json
    """
    os.makedirs(base_path, exist_ok=True)

    if file_format not in {"parquet", "csv"}:
        raise ValueError("file_format debe ser 'parquet' o 'csv'.")

    def _save_df(df: pd.DataFrame, name: str):
        path = os.path.join(base_path, f"{name}.{file_format}")
        if file_format == "parquet":
            df.to_parquet(path, index=False)
        else:
            df.to_csv(path, index=False)

    _save_df(mnq_train_scaled, "mnq_train_scaled")
    _save_df(mnq_valid_scaled, "mnq_valid_scaled")
    _save_df(mnq_test_scaled,  "mnq_test_scaled")

    # Guardar metadata (JSON serializable)
    meta_path = os.path.join(base_path, "scaling_meta.json")
    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(scale_meta, f, indent=2, ensure_ascii=False)


### **5.2.3. Calcular o cargar datasets escalados**


In [74]:
import os
import json
import joblib
import pandas as pd
from typing import Tuple, Dict, Any


def load_or_scale_mnq_datasets(
    *,
    mnq_train: pd.DataFrame,
    mnq_valid: pd.DataFrame,
    mnq_test: pd.DataFrame,
    scaler,
    base_path: str = "/content/drive/MyDrive/neural_profit/data/scaled",
    date_col: str = "date",
    target_cols: tuple[str, str] = ("delta60", "delta90"),
    file_format: str = "parquet",
    verbose: bool = True,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, Dict[str, Any], Any]:
    """
    Carga datasets escalados y scaler si existen.
    Si no existen, escala, guarda datasets, metadata y scaler entrenado.

    Muestra por pantalla qué decisión toma en cada paso.
    """
    os.makedirs(base_path, exist_ok=True)

    # Paths esperados
    train_path  = os.path.join(base_path, f"mnq_train_scaled.{file_format}")
    valid_path  = os.path.join(base_path, f"mnq_valid_scaled.{file_format}")
    test_path   = os.path.join(base_path, f"mnq_test_scaled.{file_format}")
    meta_path   = os.path.join(base_path, "scaling_meta.json")
    scaler_path = os.path.join(base_path, "scaler.joblib")

    paths = {
        "mnq_train_scaled": train_path,
        "mnq_valid_scaled": valid_path,
        "mnq_test_scaled":  test_path,
        "scaling_meta":     meta_path,
        "scaler":           scaler_path,
    }

    if verbose:
        print("Verificando existencia de datasets escalados y scaler...")
        for name, path in paths.items():
            print(f"  - {name}: {'OK' if os.path.exists(path) else 'NO EXISTE'}")

    files_exist = all(os.path.exists(p) for p in paths.values())

    # -------------------------------------------------
    # Caso 1: todo existe → cargar y salir
    # -------------------------------------------------
    if files_exist:
        if verbose:
            print("\nTodos los archivos existen.")
            print("Cargando datasets escalados y scaler entrenado...")

        if file_format == "parquet":
            mnq_train_s = pd.read_parquet(train_path)
            mnq_valid_s = pd.read_parquet(valid_path)
            mnq_test_s  = pd.read_parquet(test_path)
        else:
            mnq_train_s = pd.read_csv(train_path)
            mnq_valid_s = pd.read_csv(valid_path)
            mnq_test_s  = pd.read_csv(test_path)

        with open(meta_path, "r", encoding="utf-8") as f:
            scale_meta = json.load(f)

        scaler = joblib.load(scaler_path)

        if verbose:
            print("Carga completada. No se recalculó el escalado.")

        return mnq_train_s, mnq_valid_s, mnq_test_s, scale_meta, scaler

    # -------------------------------------------------
    # Caso 2: no existe → calcular escalado
    # -------------------------------------------------
    if verbose:
        print("\nNo se encontraron todos los archivos necesarios.")
        print("Recalculando escalado desde cero...")
        print("Ajustando scaler SOLO con mnq_train...")

    mnq_train_s, mnq_valid_s, mnq_test_s, scale_meta = scale_mnq_splits(
        mnq_train,
        mnq_valid,
        mnq_test,
        scaler=scaler,
        date_col=date_col,
        target_cols=target_cols,
    )

    if verbose:
        print("Guardando datasets escalados...")

    def _save_df(df: pd.DataFrame, path: str):
        if file_format == "parquet":
            df.to_parquet(path, index=False)
        else:
            df.to_csv(path, index=False)

    _save_df(mnq_train_s, train_path)
    _save_df(mnq_valid_s, valid_path)
    _save_df(mnq_test_s,  test_path)

    if verbose:
        print("Guardando metadata de escalado...")

    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(scale_meta, f, indent=2, ensure_ascii=False)

    if verbose:
        print("Guardando scaler entrenado...")

    joblib.dump(scaler, scaler_path)

    if verbose:
        print("Escalado completo y persistido en disco.")

    return mnq_train_s, mnq_valid_s, mnq_test_s, scale_meta, scaler


## **5.3. Aplicación**


In [82]:
scaler = choose_scaler("standard")

mnq_train_s, mnq_valid_s, mnq_test_s, scale_meta, scaler = load_or_scale_mnq_datasets(
    mnq_train=mnq_train,
    mnq_valid=mnq_valid,
    mnq_test=mnq_test,
    scaler=scaler,
    base_path="/content/drive/MyDrive/neural_profit/data/scaled",
)

Verificando existencia de datasets escalados y scaler...
  - mnq_train_scaled: OK
  - mnq_valid_scaled: OK
  - mnq_test_scaled: OK
  - scaling_meta: OK
  - scaler: OK

Todos los archivos existen.
Cargando datasets escalados y scaler entrenado...
Carga completada. No se recalculó el escalado.


# **6. Generación de ventanas deslizantes (sliding windows)**

## **6.1. Introducción conceptual**

Se generan ventanas **consecutivas y no aleatorias**, es decir, **ventanas deslizantes (sliding windows)** dentro de cada día de operación.

**1. Agrupamiento diario**

El dataset se organiza inicialmente por **día de operación bursátil**.

- Cada día contiene **421 registros consecutivos minuto a minuto**, correspondientes al horario **07:30–14:30 (America/New_York)**.
- Los splits `mnq_train`, `mnq_valid` y `mnq_test` se realizan **a nivel diario**, de modo que **no existen días compartidos entre splits**.
- Una vez realizado el split por días, las ventanas deslizantes se generan **independientemente dentro de cada día**.

---

**2. Iteración dentro del día**

Para cada día, se generan ventanas deslizantes que comienzan en el minuto $i$ y terminan en $ i + \text{window\_size} - 1 $.

- El **window size** es fijo e igual a **60 minutos**.
- El **target** se define como el valor correspondiente al **último registro de la ventana** (enfoque **seq2one**).

Ejemplo

Si `window_size = 60` y el día tiene 421 minutos:

| Iteración | Ventana usada     | Target extraído       |
|-----------|-------------------|-----------------------|
| i = 0     | registros 0-59    | target = registro 59  |
| i = 1     | registros 1-60    | target = registro 60  |
| i = 2     | registros 2-61    | target = registro 61  |
| ...       | ...               | ...                   |
| i = 361   | registros 361-420 | target = registro 420 |

Esto genera:

$$
421 - 60 + 1 = 362
$$

**ventanas por día**, todas consecutivas y **sin solapamientos entre días**.

---

**3. Estructura de las features**

- Cada ventana está compuesta por **60 registros temporales**.
- Cada registro contiene **20 features de entrada**, incluyendo:
  - indicadores técnicos,
  - interacciones entre indicadores,
  - flags temporales,
  - `minute_of_day`.

Por lo tanto, cada ventana forma una **matriz de entrada de dimensión**:

$$
60 \times 20
$$

donde:
- cada fila representa un minuto,
- cada columna representa una feature de entrada.

---

**4. Definición del target**

- El target está definido a nivel de cada fila del dataset y representa el **delta de puntos hacia adelante** (por ejemplo, a 60 o 90 minutos).
- En el enfoque **seq2one**, el modelo:
  - consume la secuencia completa de 60 minutos,
  - y **predice un único valor escalar**, correspondiente al **target del último minuto de la ventana**.

---

**5. Definición formal de cada muestra**

- **Entrada:** matriz \( 60 \times 20 \)  
- **Salida:** escalar \( 1 \times 1 \)

---

**6. Propiedades del esquema**

Este diseño garantiza que:

- no existe superposición entre días,
- las ventanas internas respetan estrictamente el orden temporal,
- no se introduce fuga de información entre splits,
- y se preserva la coherencia causal entre inputs y target.

El resultado es un dataset consistente con un enfoque **seq2one intradía**, entrenado con **sliding windows full-day** y compatible con una operativa restringida a ventanas horarias específicas.

## **6.2. Funciones para construcción de ventanas secuencia a secuencia**

### **6.2.1. Generador seq2one vectorizado**

In [83]:
import numpy as np
import pandas as pd
from typing import List, Tuple, Optional


def generate_vectorized_windows_seq2one(
    df: pd.DataFrame,
    *,
    date_col: str,
    features: List[str],
    target_col: str,
    window_size: int,
    drop_windows_with_nan: bool = True,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Genera ventanas deslizantes (sliding windows) por día bajo un esquema SEQ2ONE.

    Para cada día:
      - Se generan ventanas consecutivas de tamaño `window_size`.
      - Cada ventana se aplana en un vector 1D de tamaño:
            window_size * n_features
      - El target corresponde al valor del ÚLTIMO registro de la ventana.

    Retorna:
      X: np.ndarray con shape (n_ventanas_totales, window_size * n_features)
      y: np.ndarray con shape (n_ventanas_totales,)
    """
    if window_size <= 0:
        raise ValueError("window_size debe ser un entero positivo.")

    required_cols = [date_col] + features + [target_col]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Faltan columnas requeridas en df: {missing}")

    X, y = [], []

    # Agrupamiento diario
    for _, grupo in df.groupby(date_col):
        # índice local por día
        grupo = grupo.reset_index(drop=True)
        n = len(grupo)

        if n < window_size:
            continue

        # Sliding windows dentro del día
        for i in range(0, n - window_size + 1):
            ventana = grupo.loc[i : i + window_size - 1, features]

            if drop_windows_with_nan and ventana.isnull().values.any():
                continue

            vector = ventana.to_numpy().reshape(-1)
            target = grupo.loc[i + window_size - 1, target_col]

            if pd.isna(target):
                continue

            X.append(vector)
            y.append(target)

    return np.asarray(X), np.asarray(y)



### **6.2.2. Load/Build para SEQ2ONE**

In [84]:
import numpy as np
from pathlib import Path
from typing import List, Tuple


def prepare_or_load_seq2one_windows(
    *,
    mnq_train,
    mnq_valid,
    mnq_test,
    features: List[str],
    target_col: str,
    window_size: int,
    out_windows_train,
    out_windows_valid,
    out_windows_test,
    date_col: str = "date",
    drop_windows_with_nan: bool = True,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Genera o carga ventanas SEQ2ONE (sliding windows) para un target específico.

    Cada muestra:
      X: (window_size * n_features,)
      y: escalar

    Los arrays finales:
      X: (n_samples, window_size * n_features)
      y: (n_samples,)
    """

    def _to_path(p) -> Path:
        return p if isinstance(p, Path) else Path(str(p))

    def _ensure_parent_dir(path: Path) -> None:
        path.parent.mkdir(parents=True, exist_ok=True)

    def _load_or_build(split_name: str, df, out_path):
        out_path = _to_path(out_path)
        _ensure_parent_dir(out_path)

        if not out_path.exists():
            print(f"No existe → Generando {split_name} ({target_col}) y guardando en:")
            print(f"   {out_path}")

            X, y = generate_vectorized_windows_seq2one(
                df=df,
                date_col=date_col,
                features=features,
                target_col=target_col,
                window_size=window_size,
                drop_windows_with_nan=drop_windows_with_nan,
            )

            np.savez_compressed(out_path, X=X, y=y)
            print(f"Guardado | X: {X.shape} | y: {y.shape}")
            return X, y

        print(f"Ya existe → Cargando {split_name} ({target_col}) desde:")
        print(f"   {out_path}")

        data = np.load(out_path)
        X, y = data["X"], data["y"]
        print(f"Cargado | X: {X.shape} | y: {y.shape}")
        return X, y

    X_train, y_train = _load_or_build("train", mnq_train, out_windows_train)
    X_valid, y_valid = _load_or_build("valid", mnq_valid, out_windows_valid)
    X_test,  y_test  = _load_or_build("test",  mnq_test,  out_windows_test)

    return X_train, y_train, X_valid, y_valid, X_test, y_test

### **6.2.3. Info para SEQ2ONE**

In [85]:
import numpy as np

def xy_info_seq2one(
    horizon_min: int,
    X_train, y_train,
    X_valid, y_valid,
    X_test,  y_test,
    *,
    window_size: int,
    n_features: int,
):
    """
    Imprime información resumida para el esquema SEQ2ONE vectorizado.

    Esperado:
      X: (n_samples, window_size * n_features)
      y: (n_samples,)
    """
    print(f"Información X/y para horizonte {horizon_min} min (SEQ2ONE vectorizado):")

    expected_dim = window_size * n_features

    for name, X, y in [
        ("entrenamiento", X_train, y_train),
        ("validación",    X_valid, y_valid),
        ("testeo",        X_test,  y_test),
    ]:
        print(f"\nSet de {name}:")
        print(f"\tX shape: {X.shape}")
        print(f"\ty shape: {y.shape}")

        if X.ndim != 2:
            print("\t[AVISO] X no es 2D. Se esperaba (n_samples, window_size * n_features).")
        else:
            n_samples, dim = X.shape
            print(f"\t{n_samples} muestras/ventanas (n_samples).")
            print(f"\tDimensión por muestra: {dim} (esperado: {expected_dim}).")
            if dim != expected_dim:
                print(f"\t[AVISO] Dimensión inesperada: dim={dim} vs esperado={expected_dim}.")

        if y.ndim != 1:
            print("\t[AVISO] y no es 1D. Se esperaba (n_samples,).")
        else:
            if X.ndim == 2 and X.shape[0] != y.shape[0]:
                print(f"\t[AVISO] n_samples difiere: X={X.shape[0]} vs y={y.shape[0]}.")

        y_arr = np.asarray(y)
        if y_arr.size > 0:
            print(
                "\tDistribución y: "
                f"mean={y_arr.mean():.6f}, std={y_arr.std():.6f}, "
                f"min={y_arr.min():.6f}, max={y_arr.max():.6f}"
            )


## **6.3. Generación de ventanas (H = 60min)**

In [88]:
features_to_windows = [
      'minute_of_day', 'close', 'ema60', 'roc60', 'roc30',
       'roc30_active', 'roc20', 'roc20_active', 'mom10', 'mom10_active',
       'mom5', 'mom5_active', 'mom3', 'mom3_active', 'mom5_mom10',
       'mom5_mom10_active', 'mom3_mom10', 'mom3_mom10_active', 'mom3_mom5',
       'mom3_mom5_active']

In [90]:
X_train_60, y_train_60, X_valid_60, y_valid_60, X_test_60, y_test_60 = (
    prepare_or_load_seq2one_windows(
        mnq_train=mnq_train_s,
        mnq_valid=mnq_valid_s,
        mnq_test=mnq_test_s,
        features=features_to_windows,
        target_col="ret60",
        window_size=60,
        out_windows_train="/content/drive/MyDrive/neural_profit/data/windows_seq2one/train_delta60_ws60.npz",
        out_windows_valid="/content/drive/MyDrive/neural_profit/data/windows_seq2one/valid_delta60_ws60.npz",
        out_windows_test="/content/drive/MyDrive/neural_profit/data/windows_seq2one/test_delta60_ws60.npz",
        date_col="date",
    )
)


No existe → Generando train (ret60) y guardando en:
   /content/drive/MyDrive/neural_profit/data/windows_seq2one/train_delta60_ws60.npz
Guardado | X: (330144, 1200) | y: (330144,)
No existe → Generando valid (ret60) y guardando en:
   /content/drive/MyDrive/neural_profit/data/windows_seq2one/valid_delta60_ws60.npz
Guardado | X: (70590, 1200) | y: (70590,)
No existe → Generando test (ret60) y guardando en:
   /content/drive/MyDrive/neural_profit/data/windows_seq2one/test_delta60_ws60.npz
Guardado | X: (70952, 1200) | y: (70952,)


In [91]:
xy_info_seq2one(
    horizon_min=60,
    X_train=X_train_60, y_train=y_train_60,
    X_valid=X_valid_60, y_valid=y_valid_60,
    X_test=X_test_60,   y_test=y_test_60,
    window_size=60,
    n_features=len(features_to_windows),
)

Información X/y para horizonte 60 min (SEQ2ONE vectorizado):

Set de entrenamiento:
	X shape: (330144, 1200)
	y shape: (330144,)
	330144 muestras/ventanas (n_samples).
	Dimensión por muestra: 1200 (esperado: 1200).
	Distribución y: mean=0.006027, std=1.008703, min=-10.462454, max=10.123409

Set de validación:
	X shape: (70590, 1200)
	y shape: (70590,)
	70590 muestras/ventanas (n_samples).
	Dimensión por muestra: 1200 (esperado: 1200).
	Distribución y: mean=0.001948, std=0.640061, min=-3.788125, max=3.934559

Set de testeo:
	X shape: (70952, 1200)
	y shape: (70952,)
	70952 muestras/ventanas (n_samples).
	Dimensión por muestra: 1200 (esperado: 1200).
	Distribución y: mean=-0.011834, std=1.021581, min=-11.731744, max=17.705906


## **6.4. Generación de ventanas (H = 90min)**

In [92]:
X_train_90, y_train_90, X_valid_90, y_valid_90, X_test_90, y_test_90 = (
    prepare_or_load_seq2one_windows(
        mnq_train=mnq_train_s,
        mnq_valid=mnq_valid_s,
        mnq_test=mnq_test_s,
        features=features_to_windows,
        target_col="ret90",
        window_size=60,
        out_windows_train="/content/drive/MyDrive/neural_profit/data/windows_seq2one/train_delta90_ws60.npz",
        out_windows_valid="/content/drive/MyDrive/neural_profit/data/windows_seq2one/valid_delta90_ws60.npz",
        out_windows_test="/content/drive/MyDrive/neural_profit/data/windows_seq2one/test_delta90_ws60.npz",
        date_col="date",
    )
)


No existe → Generando train (ret90) y guardando en:
   /content/drive/MyDrive/neural_profit/data/windows_seq2one/train_delta90_ws60.npz
Guardado | X: (330144, 1200) | y: (330144,)
No existe → Generando valid (ret90) y guardando en:
   /content/drive/MyDrive/neural_profit/data/windows_seq2one/valid_delta90_ws60.npz
Guardado | X: (70590, 1200) | y: (70590,)
No existe → Generando test (ret90) y guardando en:
   /content/drive/MyDrive/neural_profit/data/windows_seq2one/test_delta90_ws60.npz
Guardado | X: (70952, 1200) | y: (70952,)


In [93]:
xy_info_seq2one(
    horizon_min=90,
    X_train=X_train_90, y_train=y_train_90,
    X_valid=X_valid_90, y_valid=y_valid_90,
    X_test=X_test_90,   y_test=y_test_90,
    window_size=60,
    n_features=len(features_to_windows),
)

Información X/y para horizonte 90 min (SEQ2ONE vectorizado):

Set de entrenamiento:
	X shape: (330144, 1200)
	y shape: (330144,)
	330144 muestras/ventanas (n_samples).
	Dimensión por muestra: 1200 (esperado: 1200).
	Distribución y: mean=0.005451, std=1.008347, min=-8.523364, max=10.233158

Set de validación:
	X shape: (70590, 1200)
	y shape: (70590,)
	70590 muestras/ventanas (n_samples).
	Dimensión por muestra: 1200 (esperado: 1200).
	Distribución y: mean=0.000713, std=0.647445, min=-3.770966, max=3.307597

Set de testeo:
	X shape: (70952, 1200)
	y shape: (70952,)
	70952 muestras/ventanas (n_samples).
	Dimensión por muestra: 1200 (esperado: 1200).
	Distribución y: mean=-0.017085, std=1.036845, min=-8.179446, max=15.131408


In [94]:
import numpy as np
import pandas as pd
from typing import List, Tuple


def inspect_random_seq2one_window(
    X: np.ndarray,
    y: np.ndarray,
    *,
    features: List[str],
    window_size: int,
    random_state: int | None = None,
) -> Tuple[pd.DataFrame, float, int]:
    """
    Selecciona una ventana aleatoria del dataset SEQ2ONE vectorizado
    y la devuelve como DataFrame temporal (window_size x n_features).

    Retorna:
    --------
    df_window : pd.DataFrame
        Ventana reconstruida con columnas = features
    target : float
        Target asociado a la ventana (último paso)
    idx : int
        Índice de la ventana seleccionada
    """
    if random_state is not None:
        rng = np.random.default_rng(random_state)
        idx = rng.integers(0, len(X))
    else:
        idx = np.random.randint(0, len(X))

    n_features = len(features)
    expected_dim = window_size * n_features

    if X.shape[1] != expected_dim:
        raise ValueError(
            f"Inconsistencia dimensional: X.shape[1]={X.shape[1]} "
            f"pero se esperaba {expected_dim} (= {window_size} x {n_features})"
        )

    x_vec = X[idx]
    target = float(y[idx])

    x_2d = x_vec.reshape(window_size, n_features)

    df_window = pd.DataFrame(
        x_2d,
        columns=features,
    )

    return df_window, target, idx


In [95]:
df_win, y_win, idx = inspect_random_seq2one_window(
    X=X_train_90,
    y=y_train_90,
    features=features_to_windows,
    window_size=60,
)

print(f"Índice de ventana: {idx}")
print(f"Target (ret90): {y_win}")

df_win


Índice de ventana: 223456
Target (ret90): -0.1624156965290272


,minute_of_day,close,ema60,roc60,roc30,roc30_active,roc20,roc20_active,mom10,mom10_active,mom5,mom5_active,mom3,mom3_active,mom5_mom10,mom5_mom10_active,mom3_mom10,mom3_mom10_active,mom3_mom5,mom3_mom5_active
0,-0.888656,0.026256,-0.880751,-0.880111,-0.775864,0.0,-0.723051,1.0,0.038914,0.0,-0.088463,1.0,-0.595879,1.0,-0.143671,0.0,-0.439601,0.0,-0.591169,1.0
1,-0.880427,0.029925,-0.593830,-0.756545,-0.573770,0.0,-0.271860,1.0,0.449693,0.0,0.229124,1.0,-0.261537,1.0,-0.404679,0.0,-0.709543,0.0,-0.682380,1.0
2,-0.872199,0.028742,-0.657818,-0.850064,-0.651021,0.0,-0.177242,1.0,0.706906,0.0,-0.160571,1.0,-0.001609,1.0,-1.158633,0.0,-0.845202,0.0,0.251457,1.0
3,-0.863971,0.030754,-0.494791,-0.769177,-0.692315,0.0,-0.024206,1.0,0.768399,0.0,-0.102824,1.0,0.704524,1.0,-1.187467,0.0,-0.452760,0.0,1.026322,1.0
4,-0.855742,0.027321,-0.720314,-0.846016,-0.811299,0.0,-0.490312,1.0,0.059441,0.0,-0.175038,1.0,-0.410177,1.0,-0.259500,0.0,-0.341596,0.0,-0.226781,1.0
5,-0.847514,0.027913,-0.655207,-0.862906,-0.668907,0.0,-0.410245,1.0,0.079968,0.0,0.200312,1.0,-0.131633,1.0,0.088137,0.0,-0.182338,0.0,-0.477592,1.0
6,-0.839286,0.025428,-0.808857,-0.935382,-0.775968,0.0,-0.526839,1.0,-0.227957,0.0,-0.550199,1.0,-0.837204,1.0,-0.230363,0.0,-0.280135,0.0,-0.158367,1.0
7,-0.831058,0.026375,-0.715854,-0.863121,-0.496892,0.0,-0.366675,1.0,-0.320226,0.0,-0.290505,1.0,-0.150243,1.0,0.160400,0.0,0.283270,0.0,0.274249,1.0
8,-0.822829,0.024244,-0.842544,-0.927076,-0.633686,0.0,-0.301251,1.0,-0.638136,0.0,-0.795411,1.0,-0.577509,1.0,0.102415,0.0,0.381007,0.0,0.547147,1.0
9,-0.814601,0.024008,-0.831768,-0.910140,-0.835457,0.0,-0.410505,1.0,-0.412656,0.0,-0.406066,1.0,-0.224629,1.0,0.174883,0.0,0.344568,0.0,0.365412,1.0


# **7. Stage Summary Report**

## **7.1. Función para crear Summary Report**

In [ ]:
import json
import os
from pathlib import Path
from datetime import datetime

import numpy as np
import joblib


def build_stage06_summary_report(
    report_path,
    *,
    window_size: int,
    scaler_type: str,
    horizons: list,
    timezone_str: str = "America/New_York",
    time_window_str: str = "08:20–08:49",
    # mapping por horizonte: {60: {"train": Path, "valid": Path, "test": Path}, 90: {...}}
    windows_paths_by_horizon: dict,
    # mapping por horizonte: {60: Path("scaler_60.pkl"), 90: Path("scaler_90.pkl")}  o { "global": Path(...) }
    scaler_paths_by_horizon: dict,
    # mapping por horizonte: {60: {"train": X_train, "valid": X_valid, "test": X_test}, 90: {...}}
    X_by_horizon: dict | None = None,
    # mapping por horizonte: {60: {"train": y_train, "valid": y_valid, "test": y_test}, 90: {...}}
    y_by_horizon: dict | None = None,
    feature_names_by_horizon: dict | None = None,  # {60: features_60, 90: features_90}
    include_scaler_stats: bool = True,
    include_y_stats: bool = True,
    verbose: bool = True,
):
    """
    Crea el report summary del stage_06 (escalado de ventanas) y lo guarda como JSON.

    Qué incluye (resumen):
    - configuración del stage (window_size, scaler_type, horizons, timezone, franja horaria)
    - shapes y conteos por split (train/valid/test)
    - validaciones (no leakage declarado, consistencia de shapes, NaNs)
    - paths a artifacts generados (npz de ventanas escaladas, scalers)
    - stats del scaler (mean/std o min/max) si se puede cargar el scaler
    - stats de y (flatten) opcional, si se pasa y_by_horizon

    Requisitos:
    - Si no pasa X_by_horizon / y_by_horizon, el reporte igual se crea usando
      los archivos .npz de windows_paths_by_horizon para leer shapes y checks.
    """

    def _p(p):
        return str(p) if p is not None else None

    def _as_path(p):
        return p if isinstance(p, Path) else Path(str(p))

    def _safe_bool(x):
        return bool(x) if x is not None else None

    def _load_npz_shapes_and_checks(npz_path: Path):
        """
        Devuelve shapes y checks básicos leyendo el .npz:
        - X_shape, y_shape
        - has_nan_X, has_nan_y
        """
        data = np.load(npz_path)
        X = data["X"]
        y = data["y"]
        return {
            "X_shape": list(X.shape),
            "y_shape": list(y.shape),
            "has_nan_X": bool(np.isnan(X).any()),
            "has_nan_y": bool(np.isnan(y).any()),
        }

    def _flatten_stats(arr: np.ndarray):
        a = np.asarray(arr).ravel()
        return {
            "mean": float(a.mean()),
            "std": float(a.std()),
            "min": float(a.min()),
            "max": float(a.max()),
        }

    def _scaler_stats(scaler, feature_names=None):
        """
        Extrae estadísticas del scaler de forma segura.
        - StandardScaler: mean_, scale_
        - MinMaxScaler: data_min_, data_max_
        """
        out = {"scaler_class": scaler.__class__.__name__}

        if hasattr(scaler, "mean_") and hasattr(scaler, "scale_"):
            means = scaler.mean_.tolist()
            scales = scaler.scale_.tolist()
            out["type"] = "standard"
            if feature_names and len(feature_names) == len(means):
                out["per_feature"] = {
                    str(fn): {"mean": float(m), "std": float(s)}
                    for fn, m, s in zip(feature_names, means, scales)
                }
            else:
                out["mean"] = [float(x) for x in means]
                out["std"] = [float(x) for x in scales]

        elif hasattr(scaler, "data_min_") and hasattr(scaler, "data_max_"):
            mins = scaler.data_min_.tolist()
            maxs = scaler.data_max_.tolist()
            out["type"] = "minmax"
            if feature_names and len(feature_names) == len(mins):
                out["per_feature"] = {
                    str(fn): {"min": float(mi), "max": float(ma)}
                    for fn, mi, ma in zip(feature_names, mins, maxs)
                }
            else:
                out["min"] = [float(x) for x in mins]
                out["max"] = [float(x) for x in maxs]
        else:
            out["type"] = "unknown"
            out["note"] = "No se encontraron atributos estándar para extraer estadísticas."

        return out

    # -------------------------
    # Construcción del reporte
    # -------------------------
    report_path = _as_path(report_path)
    report_path.parent.mkdir(parents=True, exist_ok=True)

    report = {
        "stage": "stage_06_training_dataset_construction",
        "description": "Escalado de ventanas SEQ2SEQ diarias (X: window_size×features → y: window_size) usando estadísticas del set de entrenamiento.",
        "generated_at": datetime.utcnow().isoformat(timespec="seconds") + "Z",
        "config": {
            "window_size": int(window_size),
            "scaler_type": str(scaler_type),
            "scaler_scope": "train_only",
            "horizons": [int(h) for h in horizons],
            "timezone": timezone_str,
            "time_window": time_window_str,
        },
        "datasets": {},
        "checks": {
            "no_leakage_assumed": True,
            "shape_consistency": True,
            "no_nan_after_scaling": True,
        },
        "artifacts": {
            "windows_npz": {},
            "scalers": {},
        },
        "notes": [],
    }

    # Por cada horizonte, extraer info de splits
    global_shape_ok = True
    global_nan_ok = True

    for h in horizons:
        h_key = str(int(h))
        feature_names = None
        if feature_names_by_horizon is not None and h in feature_names_by_horizon:
            feature_names = list(feature_names_by_horizon[h])

        # Paths a windows escaladas
        paths = windows_paths_by_horizon.get(h, {})
        train_p = _as_path(paths.get("train"))
        valid_p = _as_path(paths.get("valid"))
        test_p  = _as_path(paths.get("test"))

        report["artifacts"]["windows_npz"][h_key] = {
            "train": _p(train_p),
            "valid": _p(valid_p),
            "test":  _p(test_p),
        }

        # Leer shapes/checks desde npz (fuente de verdad)
        info_train = _load_npz_shapes_and_checks(train_p)
        info_valid = _load_npz_shapes_and_checks(valid_p)
        info_test  = _load_npz_shapes_and_checks(test_p)

        # Validar consistencia shape: X 3D y y 2D y window_size consistente
        def _is_expected(info):
            Xs = info["X_shape"]
            ys = info["y_shape"]
            ok = True
            ok &= (len(Xs) == 3)
            ok &= (len(ys) == 2)
            if len(Xs) == 3:
                ok &= (Xs[1] == window_size)
            if len(ys) == 2:
                ok &= (ys[1] == window_size)
            if len(Xs) == 3 and len(ys) == 2:
                ok &= (Xs[0] == ys[0])
                ok &= (Xs[1] == ys[1])
            return bool(ok)

        shape_ok = _is_expected(info_train) and _is_expected(info_valid) and _is_expected(info_test)
        global_shape_ok &= shape_ok

        nan_ok = (not info_train["has_nan_X"] and not info_train["has_nan_y"] and
                  not info_valid["has_nan_X"] and not info_valid["has_nan_y"] and
                  not info_test["has_nan_X"]  and not info_test["has_nan_y"])
        global_nan_ok &= nan_ok

        # Conteos y n_features desde shapes (train como referencia)
        X_shape_train = info_train["X_shape"]
        n_features = X_shape_train[2] if len(X_shape_train) == 3 else None

        report["datasets"][h_key] = {
            "n_features": int(n_features) if n_features is not None else None,
            "splits": {
                "train": int(info_train["X_shape"][0]),
                "valid": int(info_valid["X_shape"][0]),
                "test":  int(info_test["X_shape"][0]),
            },
            "shapes": {
                "train": {"X": info_train["X_shape"], "y": info_train["y_shape"]},
                "valid": {"X": info_valid["X_shape"], "y": info_valid["y_shape"]},
                "test":  {"X": info_test["X_shape"],  "y": info_test["y_shape"]},
            },
            "checks": {
                "shape_ok": shape_ok,
                "no_nan": nan_ok,
            },
        }

        # Stats de y (opcional): si pasaron y_by_horizon, usar eso; si no, leer y desde train npz
        if include_y_stats:
            if y_by_horizon is not None and h in y_by_horizon and "train" in y_by_horizon[h]:
                ytr = y_by_horizon[h]["train"]
                yva = y_by_horizon[h].get("valid")
                yte = y_by_horizon[h].get("test")
            else:
                # cargar desde npz (solo stats, no guardar data)
                ytr = np.load(train_p)["y"]
                yva = np.load(valid_p)["y"]
                yte = np.load(test_p)["y"]

            report["datasets"][h_key]["y_stats"] = {
                "train": _flatten_stats(ytr),
                "valid": _flatten_stats(yva),
                "test":  _flatten_stats(yte),
            }

        # Stats del scaler (opcional): cargar pkl si existe
        scaler_p = scaler_paths_by_horizon.get(h) or scaler_paths_by_horizon.get(h_key) or scaler_paths_by_horizon.get("global")
        if scaler_p is not None:
            scaler_p = _as_path(scaler_p)
            report["artifacts"]["scalers"][h_key] = _p(scaler_p)

            if include_scaler_stats and scaler_p.exists():
                try:
                    scaler_obj = joblib.load(scaler_p)
                    report["datasets"][h_key]["scaler_stats"] = _scaler_stats(scaler_obj, feature_names=feature_names)
                except Exception as e:
                    report["datasets"][h_key]["scaler_stats_error"] = str(e)
        else:
            report["artifacts"]["scalers"][h_key] = None
            report["notes"].append(f"No se proporcionó scaler_path para horizonte {h_key}.")

        # Incluir features usadas (opcional, útil para trazabilidad)
        if feature_names is not None:
            report["datasets"][h_key]["feature_names"] = feature_names

    # Checks globales
    report["checks"]["shape_consistency"] = bool(global_shape_ok)
    report["checks"]["no_nan_after_scaling"] = bool(global_nan_ok)

    # Nota sobre y no escalada (decisión de diseño)
    report["notes"].append("El target (y) se guarda sin escalar para mantener unidades en puntos (delta_pts).")
    report["notes"].append("El scaler se ajusta únicamente con X_train (aplanando n_days×window_size) para evitar leakage.")

    # Guardar
    with open(report_path, "w", encoding="utf-8") as f:
        json.dump(report, f, indent=2, ensure_ascii=False)

    if verbose:
        print(f"Report stage_06 guardado en: {report_path}")

    return report


### **7.2. Función imprimiar Summary Report**

In [ ]:
from typing import Dict, Any


def print_stage06_summary_pretty(summary: Dict[str, Any]) -> None:
    """
    Pretty print del summary dict de stage_06 (window scaling).
    """
    if not summary:
        print("Summary vacío (stage_06).")
        return

    config = summary.get("config", {})
    datasets = summary.get("datasets", {})
    checks = summary.get("checks", {})
    artifacts = summary.get("artifacts", {})

    print("\n" + "=" * 78)
    print("STAGE_06 – TRAINING DATASET CONSTRUCTION (SEQ2SEQ MNQ)")
    print("=" * 78)

    # ------------------------------------------------------------------
    # Configuración general
    # ------------------------------------------------------------------
    print("\nConfiguración")
    print("-" * 78)
    print(f"Window size        : {config.get('window_size')}")
    print(f"Scaler type        : {config.get('scaler_type')}")
    print(f"Scaler scope       : {config.get('scaler_scope')}")
    print(f"Horizontes         : {config.get('horizons')}")
    print(f"Timezone           : {config.get('timezone')}")
    print(f"Ventana horaria    : {config.get('time_window')}")

    # ------------------------------------------------------------------
    # Datasets por horizonte
    # ------------------------------------------------------------------
    for h, info in datasets.items():
        print("\n" + "-" * 78)
        print(f"Horizonte {h} minutos")
        print("-" * 78)

        print(f"Features           : {info.get('n_features')}")

        splits = info.get("splits", {})
        print("Splits (n_days)")
        print(f"  Train            : {splits.get('train')}")
        print(f"  Valid            : {splits.get('valid')}")
        print(f"  Test             : {splits.get('test')}")

        shapes = info.get("shapes", {})
        if shapes:
            print("\nShapes")
            for split_name, sh in shapes.items():
                print(f"  {split_name:<6} -> X: {sh.get('X')} | y: {sh.get('y')}")

        # Checks por horizonte
        h_checks = info.get("checks", {})
        if h_checks:
            print("\nChecks")
            print(f"  Shape OK         : {h_checks.get('shape_ok')}")
            print(f"  No NaN           : {h_checks.get('no_nan')}")

        # Estadísticas del target
        y_stats = info.get("y_stats")
        if y_stats:
            print("\nTarget (y) stats – flatten")
            for split_name, st in y_stats.items():
                print(
                    f"  {split_name:<6} -> "
                    f"mean={st.get('mean'):.6f} | "
                    f"std={st.get('std'):.6f} | "
                    f"min={st.get('min'):.6f} | "
                    f"max={st.get('max'):.6f}"
                )

        # Estadísticas del scaler
        scaler_stats = info.get("scaler_stats")
        if scaler_stats:
            print("\nScaler stats")
            print(f"  Class            : {scaler_stats.get('scaler_class')}")
            print(f"  Type             : {scaler_stats.get('type')}")

            per_feat = scaler_stats.get("per_feature")
            if per_feat:
                print("  Por feature:")
                for fname, vals in per_feat.items():
                    if "mean" in vals:
                        print(
                            f"    {fname:<20} "
                            f"mean={vals['mean']:.6f} | std={vals['std']:.6f}"
                        )
                    else:
                        print(
                            f"    {fname:<20} "
                            f"min={vals['min']:.6f} | max={vals['max']:.6f}"
                        )

    # ------------------------------------------------------------------
    # Checks globales
    # ------------------------------------------------------------------
    print("\n" + "-" * 78)
    print("Checks globales")
    print("-" * 78)
    print(f"No leakage asumido : {checks.get('no_leakage_assumed')}")
    print(f"Shape consistente  : {checks.get('shape_consistency')}")
    print(f"No NaN post-scale  : {checks.get('no_nan_after_scaling')}")

    # ------------------------------------------------------------------
    # Artifacts
    # ------------------------------------------------------------------
    print("\n" + "-" * 78)
    print("Artifacts generados")
    print("-" * 78)

    win_art = artifacts.get("windows_npz", {})
    for h, paths in win_art.items():
        print(f"Horizonte {h}:")
        print(f"  Train  : {paths.get('train')}")
        print(f"  Valid  : {paths.get('valid')}")
        print(f"  Test   : {paths.get('test')}")

    scalers = artifacts.get("scalers", {})
    if scalers:
        print("\nScalers")
        for h, p in scalers.items():
            print(f"  {h:<6} : {p}")

    # ------------------------------------------------------------------
    # Notas
    # ------------------------------------------------------------------
    notes = summary.get("notes", [])
    if notes:
        print("\n" + "-" * 78)
        print("Notas")
        print("-" * 78)
        for n in notes:
            print(f"- {n}")

    print("\n" + "=" * 78)


### **7.3. Creación de Summary Report**

In [ ]:
windows_paths_by_horizon = {
    60: {"train": OUT_WINDOWS_60_TRAIN_Z, "valid": OUT_WINDOWS_60_VALID_Z, "test": OUT_WINDOWS_60_TEST_Z},
    90: {"train": OUT_WINDOWS_90_TRAIN_Z, "valid": OUT_WINDOWS_90_VALID_Z, "test": OUT_WINDOWS_90_TEST_Z},
}

# Recomendación: un scaler por horizonte (para no sobrescribir). Si usted usa uno global, use {"global": OUT_SCALER}
scaler_paths_by_horizon = {
    60: OUT_SCALER_60,
    90: OUT_SCALER_90,
}

report = build_stage06_summary_report(
    report_path=OUT_SUMMARY,
    window_size=30,
    scaler_type="standard",
    horizons=[60, 90],
    timezone_str="America/New_York",
    time_window_str="08:20–08:49",
    windows_paths_by_horizon=windows_paths_by_horizon,
    scaler_paths_by_horizon=scaler_paths_by_horizon,
    feature_names_by_horizon={60: features_60, 90: features_90},  # opcional
    include_scaler_stats=True,
    include_y_stats=True,
    verbose=True,
)


NameError: name 'features_60' is not defined

In [ ]:
print_stage06_summary_pretty(report)

## **8. Alineamiento con libro de ML**

## Preprocesamiento y escalamiento de datos

Este paso se realiza **después del split temporal** y **antes del entrenamiento del modelo**, de acuerdo con las buenas prácticas de *Machine Learning* para series temporales financieras.

---

### Aspectos correctamente alineados

1. **Orden del proceso**
   - El split temporal se realiza **antes** del escalamiento.
   - El escalamiento se realiza **antes** del entrenamiento del modelo.

2. **Regla crítica anti-leakage**
   - El *scaler* se ajusta **exclusivamente con el conjunto de entrenamiento (TRAIN)**.
   - Los conjuntos de *validation* y *test* se transforman utilizando ese mismo *scaler*,
     sin volver a ajustarlo.

3. **Escalamiento aplicado únicamente a features**
   - Las variables de entrada (OHLCV / features) son escaladas.
   - El target (`delta_pts_H`) **no se escala**, preservando su interpretación económica en unidades de puntos.

4. **Consistencia entre horizontes**
   - Se utilizan *scalers* independientes para **H = 60** y **H = 90**.
   - No se mezclan estadísticas entre distintos horizontes de predicción.

5. **Persistencia**
   - Los *scalers* se guardan para asegurar reproducibilidad.
   - Son reutilizables en etapas posteriores de inferencia.

---

### Criterio de escalamiento

Se utiliza **StandardScaler (z-score)**, ajustado exclusivamente con el conjunto de
entrenamiento, por su adecuación a modelos sensibles a la escala.

El escalamiento se realiza **por feature de forma global**, y no de manera independiente
por jornada bursátil.

---

### Verificación automática del escalamiento

Se realizó un *sanity check* sobre el conjunto de entrenamiento escalado, verificando que:

- la media por feature sea aproximadamente 0,
- el desvío estándar por feature sea aproximadamente 1,

únicamente en el conjunto **TRAIN**.

Resultado para **H = 60**:

[OK] Escalamiento TRAIN verificado |

max |mean| = 8.2516e-16

max |std-1| = 8.8818e-15

Estos valores corresponden a error numérico de precisión flotante, confirmando que el
escalamiento fue correctamente ajustado sobre el conjunto de entrenamiento, sin
contaminación de *validation* ni *test*.
